In [9]:
from sklearn.datasets import fetch_openml
import pandas as pd

# 1. Chargement des données
vehicle_data = fetch_openml(name='vehicle', version=1, as_frame=True)

# 2. Création de la table complète (Variables explicatives + Variable cible)
df_vehicle = vehicle_data.data.copy()
df_vehicle['Class'] = vehicle_data.target

# 3. Affichage des 5 premières lignes
df_vehicle.head()

,COMPACTNESS,CIRCULARITY,DISTANCE_CIRCULARITY,RADIUS_RATIO,PR.AXIS_ASPECT_RATIO,MAX.LENGTH_ASPECT_RATIO,SCATTER_RATIO,ELONGATEDNESS,PR.AXIS_RECTANGULARITY,MAX.LENGTH_RECTANGULARITY,SCALED_VARIANCE_MAJOR,SCALED_VARIANCE_MINOR,SCALED_RADIUS_OF_GYRATION,SKEWNESS_ABOUT_MAJOR,SKEWNESS_ABOUT_MINOR,KURTOSIS_ABOUT_MAJOR,KURTOSIS_ABOUT_MINOR,HOLLOWS_RATIO,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [13]:
predictors = [col for col in df_vehicle.columns 
              if not col.lower().startswith(('skewness', 'kurtosis'))]

In [14]:
df=df_vehicle[predictors]

In [17]:
df.shape

(846, 15)

In [21]:
y=df["Class"]
X = df.drop(columns=["Class"])

In [22]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Mandatory Step: Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Pipeline: Feature Standardization + k-NN Classifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# 3. Strategy: 5-Fold Cross-Validation over a grid of k
param_grid = {'knn__n_neighbors': list(range(1, 31, 2))} # odd values to break ties

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Optimal k
best_k = grid_search.best_params_['knn__n_neighbors']
best_cv_accuracy = grid_search.best_score_
best_cv_error = 1.0 - best_cv_accuracy

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_error = 1.0 - test_accuracy

print(f"Optimal k:                      {best_k}")
print(f"Cross-Validation Error (Train): {best_cv_error:.4f}")
print(f"Test Accuracy:                  {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Test Misclassification Error:    {test_error:.4f} ({test_error * 100:.2f}%)")

Optimal k:                      3
Cross-Validation Error (Train): 0.2544
Test Accuracy:                  0.7176 (71.76%)
Test Misclassification Error:    0.2824 (28.24%)
